# R07 v2 — N01 Lesion-Aware Thesis-Style Calibrated Ensemble (Seed 42)

**Kaggle notebook:** `lesion-r07-n01-thesis-style-ensemble-s42`  
**Required Kaggle notebook outputs:** completed `lesion-r04-ssl-classification-s42`, `lesion-r05-n01-dual-ssl-featurebank-s42`, `lesion-r06-n01-rotation-simclr-feature-fusion-s42`.  
**GPU:** not required (CPU sufficient); **Internet:** OFF. **Expected runtime:** substantially below 10 hours.

Uses the **six existing N01 models**: Rotation, SimCLR, FusionConcat, FusionGatedV2, FusionGatedV3 and ResidualFusion. The two R04 classifier heads are applied to the **unmodified R05 2,048-D features**, and cross-checked against saved R04 validation predictions. Four R06 models are loaded from saved **meta-train and validation probabilities**. No encoder or fusion head is retrained.

**Strict partition roles:** original `train` trained all six base models; **`meta_train` fits temperature scalers, constrained weights and a logistic stacker**; `validation` selects the final ensemble/blend/bias; `test` is never accessed. This differs from the old thesis notebook, which learned stacking on the validation partition. All reported R07 scores are **validation scores only**. The model should be frozen before running a separate R08 locked-test inference notebook.

**v2 fix:** lesion-aware class-complete CV generated adaptively; never require every random StratifiedGroupKFold fold to contain all eight classes. Validation is scored on the complete partition, not on arbitrarily incomplete folds. If a class belongs to only one lesion group, 8-class grouped CV is mathematically impossible and is explicitly skipped; stacking then uses a predeclared regularization value and retains validation-only model selection. No locked-test access.


## 1. Setup and reproducibility

In [1]:
import os,json,hashlib,random,time,zipfile,shutil,io,math
from pathlib import Path
import numpy as np,pandas as pd,torch,joblib
from scipy.optimize import minimize_scalar
from scipy.special import softmax
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score,accuracy_score,f1_score,log_loss,confusion_matrix,classification_report

NAME='lesion-r07-n01-thesis-style-ensemble-s42';SEED=42
INPUT=Path(os.getenv('R07_INPUT_DIR','/kaggle/input' if Path('/kaggle/input').exists() else '/mnt/data/r07_test_inputs')).resolve()
OUT=Path(os.getenv('R07_OUT','/kaggle/working/'+NAME if Path('/kaggle/working').exists() else './'+NAME)).resolve()
STAGING=Path(os.getenv('R07_STAGE_DIR','/kaggle/working/.r07_input' if Path('/kaggle/working').exists() else './.r07_input')).resolve()
CLASSES=['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
SOURCES=['rotation','simclr','FusionConcat','FusionGatedV2','FusionGatedV3','ResidualFusion']
EXPECTED={'train':16490,'meta_train':2267,'validation':2186}
C_GRID=[0.01,0.1,1.0];STACK_ALPHA=[0.3,0.5,0.7,0.9]
BIAS_GRID=[-.15,-.075,0.,.075,.15];WEIGHT_SAMPLES=4096;FOLDS=5
MAX_HOURS=7.5;START=time.monotonic()
random.seed(SEED);np.random.seed(SEED)
OUT.mkdir(parents=True,exist_ok=True)
for sub in ['reports','tables','predictions','models']:(OUT/sub).mkdir(exist_ok=True)

def req(v,msg):
    if not bool(v):raise RuntimeError(msg)
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(2**20),b''):h.update(b)
    return h.hexdigest()
def jwrite(p,x):
    p=Path(p);tmp=p.with_suffix('.tmp');tmp.write_text(json.dumps(x,indent=2,allow_nan=False,default=str));tmp.replace(p)
def normalize(p):
    p=np.asarray(p,dtype=np.float64)
    req(p.ndim==2 and p.shape[1]==8 and np.isfinite(p).all(),'Invalid source probability tensor')
    p=np.maximum(p,1e-12);return p/p.sum(axis=1,keepdims=True)
def score(y,p):
    pred=np.argmax(p,axis=1)
    return {'balanced_accuracy':float(balanced_accuracy_score(y,pred)),
            'accuracy':float(accuracy_score(y,pred)),
            'macro_f1':float(f1_score(y,pred,labels=np.arange(8),average='macro',zero_division=0)),
            'nll':float(log_loss(y,normalize(p),labels=np.arange(8)))}
def tscale(p,t):return softmax(np.log(np.clip(p,1e-12,1))/float(t),axis=1)
def apply_bias(p,b):return softmax(np.log(np.clip(p,1e-12,1))+np.asarray(b)[None,:],axis=1)
def calibrate(p,y):
    obj=lambda lt: log_loss(y,tscale(p,np.exp(lt)),labels=np.arange(8))
    r=minimize_scalar(obj,bounds=(np.log(.45),np.log(3.5)),method='bounded',options={'xatol':1e-4})
    req(r.success,'Temperature optimization did not converge')
    return float(np.exp(r.x))
print(NAME,'torch',torch.__version__,'input',INPUT)

lesion-r07-n01-thesis-style-ensemble-s42 torch 2.11.0+cu130 input /kaggle/input


## 2. Resolve completed Kaggle outputs, including nested notebook ZIPs

In [2]:
def discover(root):
    found={'R04':[],'R05':[],'R06':[]}
    markers={'R04':'reports/run_summary.json','R05':'reports/final_status.json','R06':'reports/final_status.json'}
    for kind,marker in markers.items():
        for p in root.rglob(Path(marker).name):
            if not p.as_posix().endswith(marker):continue
            try:d=json.loads(p.read_text())
            except Exception:continue
            if d.get('stage')!=kind or d.get('seed')!=SEED or d.get('status')!='COMPLETED':continue
            candidate=p.parent.parent
            found[kind].append(candidate.resolve())
        found[kind]=sorted(set(found[kind]))
    return found

def unpack(zp,dst):
    dst.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(zp) as z:
        files=[x for x in z.infolist() if not x.is_dir()]
        req(sum(x.file_size for x in files)<min(12*2**30,int(shutil.disk_usage(dst).free*.85)),'Insufficient disk to unpack input')
        for ent in files:
            rel=Path(ent.filename.replace('\\','/'))
            req(not rel.is_absolute() and '..' not in rel.parts,'Unsafe archive path')
            path=(dst/rel).resolve();req(path.is_relative_to(dst.resolve()),'ZIP traversal')
            mode=(ent.external_attr>>16)&0o170000
            req(mode!=0o120000,'Symlink in input ZIP')
            path.parent.mkdir(exist_ok=True,parents=True)
            if path.exists() and path.stat().st_size==ent.file_size:continue
            with z.open(ent) as a,open(path,'wb') as b:shutil.copyfileobj(a,b,2**20)

roots=discover(INPUT)
if any(len(v)!=1 for v in roots.values()):
    archives=sorted(INPUT.rglob('*.zip'))
    for level in range(2):
        if all(len(v)==1 for v in roots.values()):break
        for zp in archives:
            dst=STAGING/hashlib.sha256(str(zp).encode()).hexdigest()[:12]
            if (dst/'.ok').exists():continue
            with zipfile.ZipFile(zp) as z:
                names=z.namelist()
                relevant=zp.name=='_output_.zip' or any(n.endswith(('reports/run_summary.json','reports/final_status.json')) for n in names) or any(n.lower().endswith('.zip') for n in names)
            if not relevant:continue
            unpack(zp,dst);(dst/'.ok').write_text('ok')
        candidates=[discover(INPUT),discover(STAGING)]
        roots={k:sorted(set(candidates[0][k]+candidates[1][k])) for k in roots}
        archives=sorted(set(archives+[p for p in STAGING.rglob('*.zip') if p.is_file()]))
for kind,v in roots.items():req(len(v)==1,f'Attach exactly one completed {kind} seed-42 Kaggle output; found: {v}')
R04,R05,R06=(roots[k][0] for k in ['R04','R05','R06'])
s04=json.loads((R04/'reports/run_summary.json').read_text())
s05=json.loads((R05/'reports/final_status.json').read_text())
s06=json.loads((R06/'reports/final_status.json').read_text())
req(s04.get('stage')=='R04' and s04.get('status')=='COMPLETED' and s04.get('seed')==SEED,'R04 not completed or wrong seed')
req(s05.get('stage')=='R05' and s05.get('status')=='COMPLETED' and s05.get('seed')==SEED,'R05 not completed or wrong seed')
req(s06.get('stage')=='R06' and s06.get('status')=='COMPLETED' and s06.get('seed')==SEED,'R06 not completed or wrong seed')
req(s06['r04_encoder_provenance']==s05['r04_provenance'],'R06/R05 R04 encoder provenance mismatch')
req(not s05['test_touched'] and not s06['test_touched'],'An input stage accessed the locked test')
req(s05['classes']==CLASSES and s05['splits']==EXPECTED,'Wrong class order or split sizes')
req(s04['n01_manifest_sha256']==s05['n01_manifest_sha256']==s06['n01_manifest_sha256'],'N01 cohort/hash mismatch')
req(s05['r04_provenance']['rotation']['encoder_sha256'] and s05['r04_provenance']['simclr']['encoder_sha256'],'Missing encoder provenance')
print('RESOLVED',{'R04':str(R04),'R05':str(R05),'R06':str(R06)})

RESOLVED {'R04': '/kaggle/input/notebooks/sudiptoroy007/lesion-r04-ssl-classification-s42/lesion-r04-ssl-classification-s42', 'R05': '/kaggle/input/notebooks/masumbillah029/lesion-r05-n01-dual-ssl-featurebank-s42/lesion-r05-n01-dual-ssl-featurebank-s42', 'R06': '/kaggle/input/notebooks/masumbillah029/lesion-r06-n01-rotation-simclr-feature-fusion-s42/lesion-r06-n01-rotation-simclr-feature-fusion-s42'}


## 3. Verify identities and assemble all six sources without retraining

In [3]:
meta={};truth={};prob={split:{} for split in ['meta_train','validation']}
for split,n in EXPECTED.items():
    item=s05['manifests'][split];f=R05/item['file']
    req(f.is_file() and sha(f)==item['sha256'],f'{split} feature manifest missing/corrupt')
    d=pd.read_csv(f,dtype='string',keep_default_na=False)
    req(len(d)==n and d.image_id.is_unique and d.label.isin(CLASSES).all(),f'Wrong {split} images/labels')
    req(d.class_index.eq(d.label.map({c:str(i) for i,c in enumerate(CLASSES)})).all(),'Class ordering mismatch')
    for key in ['image_id','group_id','source_sha256']:
        req(d[key].ne('').all(),f'{split} missing {key}')
    meta[split]=d;truth[split]=d.class_index.astype(int).to_numpy(dtype=np.int64)
for a,b in [('train','meta_train'),('train','validation'),('meta_train','validation')]:
    for k in ['image_id','group_id','source_sha256']:
        req(set(meta[a][k]).isdisjoint(set(meta[b][k])),f'{k}: overlap between {a}/{b}')
req(all(len(np.unique(truth[s]))==8 for s in ['meta_train','validation']),'Missing class in stacking/evaluation')

for source in ['rotation','simclr']:
    src=s05['r04_provenance'][source]
    ep=R04/'models'/source/'encoder_best.pt';bp=R04/'models'/source/'best.pt';vp=R04/'predictions'/source/'validation.npz'
    req(all(p.is_file() for p in [ep,bp,vp]),f'R04 {source} incomplete')
    req(sha(ep)==src['encoder_sha256'] and sha(bp)==src['best_classifier_sha256'] and sha(vp)==src['validation_prob_sha256'],f'{source} R04 hash/provenance failed')
    ck=torch.load(bp,map_location='cpu',weights_only=False)
    weights=ck['model']['classifier.weight'].detach().cpu().numpy().astype(np.float64)
    bias=ck['model']['classifier.bias'].detach().cpu().numpy().astype(np.float64)
    req(weights.shape==(8,2048) and bias.shape==(8,),f'{source} classifier head mismatch')
    for split in prob:
        item=s05['features'][split][source];p=R05/item['file']
        req(p.is_file() and sha(p)==item['sha256'],f'{split}/{source} feature file invalid')
        x=np.load(p,mmap_mode='r',allow_pickle=False)
        req(x.shape==(EXPECTED[split],2048) and np.isfinite(x).all(),f'{split}/{source} features invalid')
        prob[split][source]=softmax(np.asarray(x,dtype=np.float64)@weights.T+bias,axis=1)
    with np.load(vp,allow_pickle=False) as z:
        req(np.array_equal(z['image_id'].astype(str),meta['validation'].image_id.astype(str).to_numpy()),f'{source} R04 validation IDs mismatch')
        req(np.array_equal(z['y'].astype(int),truth['validation']),f'{source} R04 validation targets mismatch')
        req(np.array_equal(z['classes'].astype(str),CLASSES),f'{source} R04 class mismatch')
        diff=float(np.max(np.abs(prob['validation'][source]-normalize(z['proba']))))
        req(diff<.02,f'R04 {source} checkpoint reconstruction mismatch: {diff:.4f}')
    print('R04 VERIFIED',source,'validation BACC',round(score(truth['validation'],prob['validation'][source])['balanced_accuracy']*100,3),'max reconstruction difference',round(diff,6))

for source in SOURCES[2:]:
    for split in prob:
        p=R06/'predictions'/f'{source}_{split}.npz';req(p.is_file(),f'Missing {p}')
        with np.load(p,allow_pickle=False) as z:
            req(np.array_equal(z['classes'].astype(str),CLASSES),'R06 class mismatch')
            req(np.array_equal(z['image_id'].astype(str),meta[split].image_id.astype(str).to_numpy()),f'R06 {source} {split} image alignment failure')
            req(np.array_equal(z['group_id'].astype(str),meta[split].group_id.astype(str).to_numpy()),f'R06 {source} {split} group mismatch')
            req(np.array_equal(z['y'].astype(int),truth[split]),f'R06 {source} {split} label mismatch')
            req(str(z['model_id'].item())==source,f'Wrong R06 {source} model')
            probs=normalize(z['proba']);logits=np.asarray(z['logits'],dtype=np.float64)
            req(probs.shape==(EXPECTED[split],8) and logits.shape==probs.shape,'Incorrect R06 prediction dimension')
            req(np.max(abs(probs-softmax(logits,axis=1)))<.001,f'R06 {source} probabilities/logits disagree')
            prob[split][source]=probs
for split in prob:req(set(prob[split])==set(SOURCES),'Missing ensemble source')
print('PREFLIGHT PASSED: six aligned prediction sources; no test-set inputs read')

R04 VERIFIED rotation validation BACC 57.717 max reconstruction difference 0.001828
R04 VERIFIED simclr validation BACC 57.562 max reconstruction difference 0.001351
PREFLIGHT PASSED: six aligned prediction sources; no test-set inputs read


## 4. Temperature calibration and safe lesion-group CV (adaptive to class group counts)

In [4]:
y_meta=truth['meta_train'];y_val=truth['validation']
g_meta=meta['meta_train'].group_id.astype(str).to_numpy()
g_val=meta['validation'].group_id.astype(str).to_numpy()

def group_support(y,g):
    return {CLASSES[c]:int(len(np.unique(g[y==c]))) for c in range(8)}

def safe_group_cv(y,g,max_folds=5,seed=42):
    y=np.asarray(y,dtype=np.int64);g=np.asarray(g,dtype=str)
    req(len(y)==len(g) and len(np.unique(y))==8,'Invalid grouped-CV input')
    support=group_support(y,g);k=min(max_folds,*support.values())
    if k<2:
        return [],{'mode':'NO_CLASS_COMPLETE_GROUP_CV','folds':0,'group_support':support,
                   'reason':'A class has fewer than two distinct lesion groups; full-class grouped CV is impossible.'}
    groups,inv=np.unique(g,return_inverse=True)
    labels=np.full(len(groups),-1,dtype=np.int64)
    sizes=np.bincount(inv)
    pure=True
    for j in range(len(groups)):
        unique=np.unique(y[inv==j])
        if len(unique)!=1:pure=False;break
        labels[j]=unique[0]
    if pure:
        # Exact class-stratified assignment at GROUP level, including very rare classes.
        rng=np.random.default_rng(seed);allocation=np.full(len(groups),-1,dtype=np.int64)
        class_load=np.zeros((8,k),dtype=np.int64);total_load=np.zeros(k,dtype=np.int64)
        for c in np.argsort([np.sum(labels==j) for j in range(8)]):
            ids=np.flatnonzero(labels==c);rng.shuffle(ids)
            ids=ids[np.argsort(-sizes[ids],kind='stable')]
            for j in ids:
                tie=rng.random(k)*1e-6
                f=min(range(k),key=lambda f:(class_load[c,f],total_load[f],tie[f]))
                allocation[j]=f;class_load[c,f]+=sizes[j];total_load[f]+=sizes[j]
        folds=[(np.flatnonzero(allocation[inv]!=i),np.flatnonzero(allocation[inv]==i)) for i in range(k)]
        mode='CLASS_BALANCED_GROUP_ASSIGNMENT'
    else:
        # Groups with mixed diagnoses require an iterative stratifier. Only accept valid folds.
        folds=[];mode='STRATIFIED_GROUP_KFOLD_RETRY'
        for n in range(k,1,-1):
            for attempt in range(200):
                candidate=list(StratifiedGroupKFold(n_splits=n,shuffle=True,random_state=seed+attempt)
                               .split(np.zeros(len(y)),y,g))
                if all(len(np.unique(y[tr]))==8 and len(np.unique(y[te]))==8 for tr,te in candidate):
                    folds=candidate;break
            if folds:break
        if not folds:
            return [],{'mode':'NO_CLASS_COMPLETE_GROUP_CV','folds':0,'group_support':support,
                       'reason':'Mixed-label groups: no class-complete lesion-disjoint CV split found after 200 attempts per fold count.'}
    check=np.zeros(len(y),dtype=np.int64)
    for tr,te in folds:
        req(len(tr)>0 and len(te)>0 and set(g[tr]).isdisjoint(set(g[te])),'Lesion crossed CV split')
        req(len(np.unique(y[tr]))==8 and len(np.unique(y[te]))==8,'CV missing a diagnostic class')
        check[te]+=1
    req(np.all(check==1),'Each meta-training sample must be held out exactly once in grouped CV')
    return folds,{'mode':mode,'folds':len(folds),'group_support':support,
                  'fold_sizes':[int(len(te)) for _,te in folds]}

cv_meta,CV_INFO=safe_group_cv(y_meta,g_meta,FOLDS,SEED)
print('META TRAINING grouped-CV audit:',json.dumps(CV_INFO))
print('VALIDATION group support:',group_support(y_val,g_val),'— evaluated as one complete held-out partition')
jwrite(OUT/'reports/group_cv_audit.json',{'meta_train':CV_INFO,'validation_group_support':group_support(y_val,g_val),'validation_evaluated_as_full_partition':True})

# Temperatures are fitted ONLY on meta_train; meta CV refits them within every training fold.
temps={s:calibrate(prob['meta_train'][s],y_meta) for s in SOURCES}
cal={part:{s:tscale(prob[part][s],temps[s]) for s in SOURCES} for part in prob}
calrows=[{'source':s,'temperature':temps[s],
          'meta_nll_before':score(y_meta,prob['meta_train'][s])['nll'],
          'meta_nll_after':score(y_meta,cal['meta_train'][s])['nll'],
          'validation_bacc':score(y_val,prob['validation'][s])['balanced_accuracy']} for s in SOURCES]
pd.DataFrame(calrows).to_csv(OUT/'tables/temperature_scaling.csv',index=False)
print(pd.DataFrame(calrows).round(5).to_string(index=False))

def features(d):return np.concatenate([np.log(np.clip(d[s],1e-12,1.)) for s in SOURCES],axis=1)
def make_stack(C):
    return Pipeline([('scaler',StandardScaler()),('clf',LogisticRegression(
        C=C,class_weight='balanced',solver='lbfgs',max_iter=3000,random_state=SEED))])
def aligned(model,x):
    z=model.predict_proba(x);ids=model.named_steps['clf'].classes_.astype(int)
    req(np.array_equal(ids,np.arange(8)),'Stacker is missing a diagnostic class')
    return normalize(z)
def complete_fold_stats(y,p,cv):
    req(len(cv)>0,'No class-complete CV available')
    pred=np.argmax(p,axis=1)
    vals=np.array([balanced_accuracy_score(y[te],pred[te]) for _,te in cv])
    return {'cv_mean_bacc':float(vals.mean()),'cv_sd_bacc':float(vals.std()),
            'cv_min_bacc':float(vals.min()),'cv_pooled_bacc':float(balanced_accuracy_score(y,pred))}

META TRAINING grouped-CV audit: {"mode": "CLASS_BALANCED_GROUP_ASSIGNMENT", "folds": 5, "group_support": {"MEL": 132, "NV": 717, "BCC": 142, "AK": 33, "BKL": 135, "DF": 12, "VASC": 20, "SCC": 24}, "fold_sizes": [453, 454, 454, 453, 453]}
VALIDATION group support: {'MEL': 117, 'NV': 764, 'BCC': 117, 'AK': 33, 'BKL': 138, 'DF': 6, 'VASC': 17, 'SCC': 24} — evaluated as one complete held-out partition
        source  temperature  meta_nll_before  meta_nll_after  validation_bacc
      rotation      1.42674          0.89729         0.85214          0.57717
        simclr      2.48793          1.28027         0.91843          0.57562
  FusionConcat      1.84106          1.06732         0.87941          0.58140
 FusionGatedV2      2.07582          1.29016         0.94753          0.58359
 FusionGatedV3      1.84788          1.15235         0.91418          0.57827
ResidualFusion      1.95072          1.21350         0.92594          0.60185


## 5. Constrained weighted averages (meta-trained; safe per-class lesion-fold diagnostics)

In [5]:
base=np.stack([cal['meta_train'][s] for s in SOURCES]).astype('float32')
valbase=np.stack([cal['validation'][s] for s in SOURCES]).astype('float32')
rng=np.random.default_rng(SEED)
W=[np.eye(len(SOURCES)),np.ones((1,len(SOURCES)))/len(SOURCES)]
for concentration in [.4,1.,3.]:W.append(rng.dirichlet(np.full(len(SOURCES),concentration),size=WEIGHT_SAMPLES//3))
for a,b in [('simclr','ResidualFusion'),('rotation','simclr'),('rotation','ResidualFusion')]:
    for t in [.25,.5,.75]:
        v=np.zeros(len(SOURCES));v[SOURCES.index(a)]=t;v[SOURCES.index(b)]+=1-t;W.append(v[None,:])
W=np.concatenate(W).astype('float32')
req(np.allclose(W.sum(axis=1),1) and (W>=0).all(),'Invalid simplex weights')
rows=[]
for offset in range(0,len(W),64):
    w=W[offset:offset+64]
    candidates=np.einsum('snc,ks->knc',base,w,optimize=True).argmax(axis=2)
    pooled=np.stack([(candidates[:,y_meta==c]==c).mean(axis=1) for c in range(8)]).mean(axis=0)
    if cv_meta:
        fvals=np.stack([np.stack([(candidates[:,te[y_meta[te]==c]]==c).mean(axis=1)
                                    for c in range(8)]).mean(axis=0) for _,te in cv_meta],axis=1)
        mean=fvals.mean(1);sd=fvals.std(1)
    else:
        mean=pooled;sd=np.zeros(len(w))
    for j in range(len(w)):
        rows.append({'idx':offset+j,'meta_bacc':float(pooled[j]),'fold_mean_bacc':float(mean[j]),
                     'fold_sd_bacc':float(sd[j]),'robust_score':float(pooled[j]-.3*sd[j])})
wdf=pd.DataFrame(rows).sort_values(['robust_score','meta_bacc','fold_sd_bacc'],ascending=[False,False,True]).reset_index(drop=True)
wdf.head(200).to_csv(OUT/'tables/weighted_meta_group_stability_top200.csv',index=False)
selected_weights=[];selected_weight_rows=[]
for _,row in wdf.iterrows():
    w=W[int(row.idx)]
    if not any(np.max(np.abs(w-x))<.035 for x in selected_weights):
        selected_weights.append(w);selected_weight_rows.append(row)
    if len(selected_weights)>=5:break
req(selected_weights,'Failed to select meta-trained ensemble weights')
for i,(w,row) in enumerate(zip(selected_weights,selected_weight_rows)):
    print('WEIGHTED',i,'meta BACC',round(100*float(row.meta_bacc),2),
          'fold SD',round(100*float(row.fold_sd_bacc),2),dict(zip(SOURCES,np.round(w,3))))

WEIGHTED 0 meta BACC 57.93 fold SD 3.61 {'rotation': np.float32(0.501), 'simclr': np.float32(0.001), 'FusionConcat': np.float32(0.109), 'FusionGatedV2': np.float32(0.002), 'FusionGatedV3': np.float32(0.001), 'ResidualFusion': np.float32(0.386)}
WEIGHTED 1 meta BACC 57.97 fold SD 4.37 {'rotation': np.float32(0.5), 'simclr': np.float32(0.0), 'FusionConcat': np.float32(0.0), 'FusionGatedV2': np.float32(0.0), 'FusionGatedV3': np.float32(0.0), 'ResidualFusion': np.float32(0.5)}
WEIGHTED 2 meta BACC 57.75 fold SD 4.05 {'rotation': np.float32(0.509), 'simclr': np.float32(0.107), 'FusionConcat': np.float32(0.02), 'FusionGatedV2': np.float32(0.003), 'FusionGatedV3': np.float32(0.0), 'ResidualFusion': np.float32(0.36)}
WEIGHTED 3 meta BACC 58.12 fold SD 5.33 {'rotation': np.float32(0.644), 'simclr': np.float32(0.077), 'FusionConcat': np.float32(0.075), 'FusionGatedV2': np.float32(0.038), 'FusionGatedV3': np.float32(0.038), 'ResidualFusion': np.float32(0.128)}
WEIGHTED 4 meta BACC 58.08 fold SD 5

## 6. Stacking: genuine lesion-disjoint OOF CV; safe predeclared-C fallback when impossible

In [6]:
cvrows=[]
if cv_meta:
    # Refit fold-specific calibrators inside each training fold, then predict held-out groups.
    fold_data=[]
    for tr,va in cv_meta:
        ft={s:calibrate(prob['meta_train'][s][tr],y_meta[tr]) for s in SOURCES}
        dtr={s:tscale(prob['meta_train'][s][tr],ft[s]) for s in SOURCES}
        dva={s:tscale(prob['meta_train'][s][va],ft[s]) for s in SOURCES}
        fold_data.append((tr,va,features(dtr),features(dva)))
    for C in C_GRID:
        oof=np.zeros((len(y_meta),8),dtype=np.float64)
        for tr,va,xtr,xva in fold_data:
            model=make_stack(C);model.fit(xtr,y_meta[tr])
            oof[va]=aligned(model,xva)
        req(np.all(oof.sum(axis=1)>.9999),'Incomplete stacker OOF predictions')
        m=score(y_meta,oof);f=complete_fold_stats(y_meta,oof,cv_meta)
        cvrows.append({'C':C,**m,**f,'robust_score':m['balanced_accuracy']-.3*f['cv_sd_bacc']})
    cdf=pd.DataFrame(cvrows).sort_values(['robust_score','balanced_accuracy','macro_f1'],ascending=False).reset_index(drop=True)
    BEST_C=float(cdf.iloc[0].C);C_SELECTION='META_GROUP_OOF'
    cdf.to_csv(OUT/'tables/stacking_meta_group_cv.csv',index=False)
    print('OOF selection:',cdf.round(5).to_string(index=False))
else:
    # No class-complete CV exists if a rare diagnosis has only one independent lesion group.
    # Never train a classifier on a fold missing that class or compute a misleading fold BACC.
    BEST_C=.1;C_SELECTION='PREDECLARED_FALLBACK_NO_CLASS_COMPLETE_GROUP_CV'
    pd.DataFrame([{'C':BEST_C,'note':CV_INFO['reason']}]).to_csv(OUT/'tables/stacking_meta_group_cv.csv',index=False)
    print('WARNING: No class-complete lesion CV. Fixed stack regularization C=0.1. Final model still selected on validation.')
STACK=make_stack(BEST_C);STACK.fit(features(cal['meta_train']),y_meta)
stack_val=aligned(STACK,features(cal['validation']))
joblib.dump(STACK,OUT/'models/stacking_meta_model.joblib',compress=3)
print('Stack C:',BEST_C,'selection:',C_SELECTION,'validation BACC:',round(100*score(y_val,stack_val)['balanced_accuracy'],3))

OOF selection:    C  balanced_accuracy  accuracy  macro_f1     nll  cv_mean_bacc  cv_sd_bacc  cv_min_bacc  cv_pooled_bacc  robust_score
0.01            0.61681   0.64314   0.50209 1.04462       0.61909     0.03481      0.58133         0.61681       0.60636
0.10            0.59776   0.64270   0.50161 1.01585       0.60007     0.02621      0.56499         0.59776       0.58989
1.00            0.56919   0.64094   0.49587 1.06685       0.57074     0.03497      0.51848         0.56919       0.55870
Stack C: 0.01 selection: META_GROUP_OOF validation BACC: 57.926


## 7. Validation-only selection: weighted averaging, anchored stacking, small class-bias grid

In [7]:
candidates={};specs={}
def register(name,p,spec):
    req(name not in candidates,'Duplicate candidate name')
    candidates[name]=normalize(p);specs[name]=spec
for s in SOURCES:
    register(s,prob['validation'][s],{'kind':'source','source':s,'calibrated':False})
    register('Cal_'+s,cal['validation'][s],{'kind':'source','source':s,'calibrated':True})
register('SSLMean',.5*(cal['validation']['rotation']+cal['validation']['simclr']),{'kind':'mean','sources':['rotation','simclr']})
register('AllMean',np.mean(valbase,axis=0),{'kind':'mean','sources':SOURCES})
for i,w in enumerate(selected_weights):
    register(f'WeightedCV_{i}',np.einsum('snc,s->nc',valbase,w,optimize=True),{'kind':'weighted','weights':w.tolist()})
anchors={'SimCLR':cal['validation']['simclr'],
         'ResidualFusion':cal['validation']['ResidualFusion'],
         'SSLMean':candidates['SSLMean'],'WeightedCV':candidates['WeightedCV_0']}
register('Stacking',stack_val,{'kind':'stacking','C':BEST_C})
for name,p in anchors.items():
    for alpha in STACK_ALPHA:
        register(f'StackBlend_{name}_a{alpha:g}',(1-alpha)*p+alpha*stack_val,
                 {'kind':'stack_blend','anchor':name,'alpha':alpha,'C':BEST_C})
rows=[]
for name,p in candidates.items():rows.append({'name':name,**score(y_val,p)})
rank=pd.DataFrame(rows).sort_values(['balanced_accuracy','macro_f1','nll'],ascending=[False,False,True]).reset_index(drop=True)
rank.to_csv(OUT/'tables/validation_candidates.csv',index=False)
champion=str(rank.iloc[0]['name']);initial=score(y_val,candidates[champion])['balanced_accuracy']
print('Top non-bias candidate:',champion,round(initial*100,3),'%')

# Small regularized coordinate search on validation; document selection optimism.
chosen=np.zeros(8,dtype=np.float64);basep=candidates[champion]
def bias_obj(b):
    p=apply_bias(basep,b);m=score(y_val,p)
    return m['balanced_accuracy']-.003*np.mean(np.abs(b))
for _ in range(2):
    for k in range(8):
        best=(bias_obj(chosen),-abs(chosen[k]),chosen[k])
        for v in BIAS_GRID:
            trial=chosen.copy();trial[k]=v
            obj=(bias_obj(trial),-abs(v),v)
            if obj[:2]>best[:2]:best=obj
        chosen[k]=best[2]
biasp=apply_bias(basep,chosen);biasm=score(y_val,biasp)
use_bias=bool(biasm['balanced_accuracy']>=initial+.003)
if use_bias:
    register('Selected_ClassBias',biasp,{'kind':'biased','base':champion,'offsets':chosen.tolist()})
    rank=pd.concat([rank,pd.DataFrame([{'name':'Selected_ClassBias',**biasm}])],ignore_index=True)
rank=rank.sort_values(['balanced_accuracy','macro_f1','nll'],ascending=[False,False,True]).reset_index(drop=True)
selected=str(rank.iloc[0]['name'])
rank.to_csv(OUT/'tables/validation_candidates.csv',index=False)
print('Validation-selected:',selected,'\n',rank.head(12).round(5).to_string(index=False))
print('Bias offsets:',chosen.tolist(),'BACC:',round(biasm['balanced_accuracy']*100,3),'accepted:',use_bias)

Top non-bias candidate: StackBlend_WeightedCV_a0.3 61.025 %
Validation-selected: Selected_ClassBias 
                           name  balanced_accuracy  accuracy  macro_f1     nll
            Selected_ClassBias            0.62043   0.71180   0.54793 0.81072
    StackBlend_WeightedCV_a0.3            0.61025   0.71592   0.54087 0.80041
    StackBlend_WeightedCV_a0.5            0.60990   0.70952   0.53096 0.83544
StackBlend_ResidualFusion_a0.5            0.60507   0.71729   0.53480 0.85665
                  WeightedCV_1            0.60489   0.73102   0.55862 0.77035
                  WeightedCV_2            0.60369   0.72964   0.55628 0.76724
       StackBlend_SSLMean_a0.5            0.60281   0.70586   0.52413 0.83802
StackBlend_ResidualFusion_a0.3            0.60278   0.72278   0.54691 0.83240
                  WeightedCV_0            0.60266   0.72873   0.55427 0.76773
    StackBlend_WeightedCV_a0.7            0.60211   0.69808   0.51558 0.88019
            Cal_ResidualFusion          

## 8. Freeze configuration, evaluate validation-only results, and save R08-ready package

In [8]:
final=candidates[selected];pred=final.argmax(1)
results=score(y_val,final)
base_ref=score(y_val,prob['validation']['ResidualFusion'])['balanced_accuracy']
cm=confusion_matrix(y_val,pred,labels=np.arange(8))
pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(OUT/'tables/final_validation_confusion_matrix.csv')
pd.DataFrame(classification_report(y_val,pred,labels=np.arange(8),target_names=CLASSES,output_dict=True,zero_division=0)).T.to_csv(OUT/'tables/final_validation_class_report.csv')
np.savez_compressed(OUT/'predictions/final_validation.npz',image_id=meta['validation'].image_id.astype(str).to_numpy(),
                    group_id=g_val,y=y_val.astype('int32'),proba=final.astype('float32'),classes=np.array(CLASSES),
                    method=np.array(selected))
for s in SOURCES:
    np.savez_compressed(OUT/'predictions'/f'{s}_meta_train_validation.npz',
        meta_image_id=meta['meta_train'].image_id.astype(str).to_numpy(),val_image_id=meta['validation'].image_id.astype(str).to_numpy(),
        meta_y=y_meta.astype('int32'),val_y=y_val.astype('int32'),
        meta_proba=prob['meta_train'][s].astype('float32'),val_proba=prob['validation'][s].astype('float32'))

# Cluster bootstrap is exploratory: model/configuration selection has already used validation labels.
groups_by_class={k:np.unique(g_val[y_val==k]) for k in range(8)}
consistent=all(len(np.unique(y_val[g_val==g]))==1 for g in np.unique(g_val))
if consistent:
    rng=np.random.default_rng(SEED+999);boot=[];baseline=prob['validation']['ResidualFusion']
    index_by_group={g:np.flatnonzero(g_val==g) for g in np.unique(g_val)}
    for _ in range(1000):
        drawn=[rng.choice(groups_by_class[k],size=len(groups_by_class[k]),replace=True) for k in range(8)]
        ids=np.concatenate([index_by_group[g] for groups in drawn for g in groups])
        d=balanced_accuracy_score(y_val[ids],final[ids].argmax(1))-balanced_accuracy_score(y_val[ids],baseline[ids].argmax(1))
        boot.append(d)
    ci=np.quantile(boot,[.025,.975]).tolist()
else:ci=None

frozen={'name':NAME,'stage':'R07','seed':SEED,'status':'COMPLETED','classes':CLASSES,'sources':SOURCES,
        'n01_manifest_sha256':s05['n01_manifest_sha256'],
        'r04_encoder_provenance':s05['r04_provenance'],
        'r05_feature_sha256':{p:{s:s05['features'][p][s]['sha256'] for s in ['rotation','simclr']} for p in ['meta_train','validation']},
        'r06_status_sha256':sha(R06/'reports/final_status.json'),
        'source_partition':'train','temperature_fit_partition':'meta_train','stacker_fit_partition':'meta_train',
        'weighted_selection_partition':'meta_train','final_selection_partition':'validation','test_touched':False,
        'group_cv_info':CV_INFO,'stacking_C_selection':C_SELECTION,
        'temperatures':temps,'stacking_C':BEST_C,'stacking_feature_transform':'concatenate per-source log calibrated probabilities',
        'candidate_name':selected,'candidate_spec':specs[selected],
        'all_specs':specs,'class_bias_grid':BIAS_GRID,
        'validation_n':len(y_val),'validation_metrics':results,
        'validation_residual_bacc':base_ref,'validation_difference_vs_residual':results['balanced_accuracy']-base_ref,
        'selection_aware_exploratory_cluster_bootstrap_diff_95ci':ci,
        'notes':'No test access. Model search and class bias used validation; estimated validation improvement is selection-optimistic. Freeze and run separate R08 locked-test evaluation.'}
serialized=json.dumps(frozen,sort_keys=True,allow_nan=False,default=str)
frozen['frozen_config_sha256']=hashlib.sha256(serialized.encode()).hexdigest()
jwrite(OUT/'reports/final_ensemble_config.json',frozen)
jwrite(OUT/'reports/final_status.json',{'stage':'R07','seed':SEED,'status':'COMPLETED',
    'selected_model':selected,'n01_manifest_sha256':s05['n01_manifest_sha256'],'test_touched':False,
    'frozen_config_sha256':frozen['frozen_config_sha256'],'validation_metrics':results})
pd.DataFrame([{'method':selected,**results,'baseline_residual_bacc':base_ref,'difference_vs_residual':results['balanced_accuracy']-base_ref,
               'cluster_bootstrap_diff_low':ci[0] if ci is not None else np.nan,
               'cluster_bootstrap_diff_high':ci[1] if ci is not None else np.nan}]).to_csv(OUT/'tables/final_validation_summary.csv',index=False)

archive=OUT.parent/(NAME+'.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=5) as z:
    for p in OUT.rglob('*'):
        if p.is_file():z.write(p,arcname=NAME+'/'+str(p.relative_to(OUT)))
print('FINAL R07 VALIDATION (NOT TEST):',selected)
print('BACC %.3f%% | Accuracy %.3f%% | Macro-F1 %.3f%%'%(100*results['balanced_accuracy'],100*results['accuracy'],100*results['macro_f1']))
print('Difference versus existing R06 ResidualFusion: %+.3f percentage points'%(100*(results['balanced_accuracy']-base_ref)))
print('Exploratory group-bootstrap 95%% CI for the difference:',ci)
print('FROZEN CONFIG',OUT/'reports/final_ensemble_config.json','ZIP',archive)

FINAL R07 VALIDATION (NOT TEST): Selected_ClassBias
BACC 62.043% | Accuracy 71.180% | Macro-F1 54.793%
Difference versus existing R06 ResidualFusion: +1.859 percentage points
Exploratory group-bootstrap 95%% CI for the difference: [-0.007509042671420943, 0.04719363071068866]
FROZEN CONFIG /kaggle/working/lesion-r07-n01-thesis-style-ensemble-s42/reports/final_ensemble_config.json ZIP /kaggle/working/lesion-r07-n01-thesis-style-ensemble-s42.zip
